In [1]:
import json
import pandas as pd

In [2]:
def data_load(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            appid, info = next(iter(item.items()))
            info['appid'] = appid
            data.append(info)
    return data

In [3]:
def df_cleansing(data_to_df):
    df = pd.DataFrame(data_to_df)
    cols_to_drop = ['header_image', 'reviews', 'supported_languages', 'support_info', 'game_link', 'appid', 'demos', 'ext_user_account_notice', 'drm_notice']
    for col in cols_to_drop:
        if col in df.columns:
            df = df.drop(col, axis=1)

    pattern = r'<[^>]*>'
    df['about_the_game'] = df['about_the_game'].str.replace(pattern, ' ', regex=True).replace(r'\n', ' ', regex=True)

    df['developers'] = df['developers'].apply(lambda x: " ".join(x) if isinstance(x, list) else x)
    df['publishers'] = df['publishers'].apply(lambda x: " ".join(x) if isinstance(x, list) else x)

    # Extract description values from genres and categories
    df['genres'] = df['genres'].apply(
        lambda x: ", ".join([d.get('description', '') for d in x]) if isinstance(x, list) else x
    )
    df['categories'] = df['categories'].apply(
        lambda x: ", ".join([d.get('description', '') for d in x]) if isinstance(x, list) else x
    )

    def extract_year(x):
        if isinstance(x, dict):
            date_str = x.get('date', '')
            try:
                return pd.to_datetime(date_str).year
            except:
                return None
        return None
    
    # Liczba DLC (jeśli pole to lista, zliczamy elementy; jeśli nie, 0)
    df['dlc_count'] = df['dlc'].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    # Liczba Osiągnięć (wyciągnięcie 'total' ze słownika)
    df['achievements_count'] = df['achievements'].apply(
        lambda x: x.get('total', 0) if isinstance(x, dict) else 0
    )

    # Usuwamy stare kolumny po ekstrakcji
    df = df.drop(['dlc', 'achievements'], axis=1)

    df['release_year'] = df['release_date'].apply(extract_year)
    df = df.drop('release_date', axis=1)

    df['is_free'] = df['is_free'].map(lambda x: int(x == True))

    df.loc[df['is_free'] == 1, 'genres'] = (
        df['genres']
        .str.replace(r',\s*Free To Play|Free To Play\s*,?', '', regex=True)
    )

    if 'controller_support' in df.columns:
        df['controller_support'] = df['controller_support'].fillna(0)
        df['controller_support'] = df['controller_support'].astype(str).str.replace('full', '1')

    if 'recommendations' in df.columns:
        df['recommendations'] = df['recommendations'].apply(
            lambda x: x.get('total', 0) if isinstance(x, dict) else 0
        )
    
    df['price_overview'] = df['price_overview'].apply(
        lambda x: float(x.get('initial', 0)) if isinstance(x, dict) else 0.0
    )
    return df

In [5]:
data_example = data_load(file_path='../data/games_informations/example.jsonl')
dataframe = df_cleansing(data_to_df=data_example)

dataframe.to_excel('ex.xlsx')